**1. Mounting Google Drive in Colab**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**2. Install library**

In [ ]:
!pip install opencv-python pillow scikit-image numpy matplotlib
!pip install rembg onnxruntime
!pip install tensorflow
!pip install scikit-image networkx scipy openpyxl

**3. Import library**

In [ ]:
import os
import math

import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf

from PIL import Image
from rembg import remove
from skimage import exposure, data
from skimage.filters import frangi, sato
from skimage.morphology import disk, erosion

**4. Set input/output paths and ensure segmentation folder exists**

In [ ]:
# Update this path according to the location of the dataset
base_folder = '/content/drive/MyDrive/In-Ovo_Sexing_Duck_Eggs'

# Input folder containing preprocessed images
preprocessing_folder = os.path.join(
    base_folder,
    'Preprocessing'
)

# Output folder for segmentation results
segmentation_folder = os.path.join(
    base_folder,
    'Segmentation'
)

# Create output folder if it does not exist
os.makedirs(
    segmentation_folder,
    exist_ok=True
)


**5. CLAHE (Contrast Limited Adaptive Histogram Equalization)**

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import os
import math

def plotHistCDF(image, subplot_index, title):
    hist = cv.calcHist([image],[0],None,[256],[0,256])
    cdf = hist.cumsum()
    cdfNorm = cdf * float(hist.max()) / cdf.max()
    plt.subplot(subplot_index[0], subplot_index[1], subplot_index[2])
    plt.plot(hist, color='red')
    plt.plot(cdfNorm, color='blue')
    plt.xlim([0,256])
    plt.xlabel(title)

def claheAll(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)

    files = sorted([f for f in os.listdir(input_folder) if f.lower().endswith('.png')])
    total_input = len(files)
    total_processed = 0
    results = []

    # Process all files
    for filename in files:
        imgPath = os.path.join(input_folder, filename)
        img = cv.imread(imgPath, cv.IMREAD_GRAYSCALE)
        if img is None:
            continue

        # Apply CLAHE
        claheObj = cv.createCLAHE(clipLimit=5, tileGridSize=(8,8))
        claheImg = claheObj.apply(img)

        # Save CLAHE result
        cv.imwrite(os.path.join(output_folder, f"{filename}"), claheImg)

        total_processed += 1
        results.append((filename, img.shape, claheImg))

if __name__ == '__main__':
    input_folder = os.path.join(preprocessing_folder, 'Multiplication')
    output_folder = os.path.join(segmentation_folder, 'CLAHE_Day-5')
    claheAll(input_folder, output_folder)

**6. Morphological Opening**

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import os

def plotHist(image):
    hist = cv.calcHist([image],[0],None,[256],[0,256])
    cdf = hist.cumsum()
    cdfNorm = cdf * float(hist.max()) / cdf.max()
    plt.plot(hist, color='red')
    plt.plot(cdfNorm, color='blue')
    plt.xlim([0,256])

def process_all_images(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)

    images = sorted([f for f in os.listdir(input_folder) if f.lower().endswith('.png')])
    kernel = cv.getStructuringElement(cv.MORPH_ELLIPSE,(5,5))
    results = []

    for fname in images:
        in_path = os.path.join(input_folder, fname)
        img = cv.imread(in_path, cv.IMREAD_GRAYSCALE)
        if img is None:
            continue

        # Morphological Opening
        morph_open = cv.morphologyEx(img, cv.MORPH_OPEN, kernel)

        out_path = os.path.join(output_folder, fname)
        cv.imwrite(out_path, morph_open)

        results.append((fname, morph_open))

if __name__ == '__main__':
    input_folder = os.path.join(segmentation_folder, 'CLAHE_Day-5')
    output_folder = os.path.join(segmentation_folder, 'Morphological_Opening_Day-5')
    process_all_images(input_folder, output_folder)

**7. Adaptive Threshold**

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import os

def plotHist(image):
    hist = cv.calcHist([image],[0],None,[256],[0,256])
    cdf = hist.cumsum()
    cdfNorm = cdf * float(hist.max()) / cdf.max()
    plt.plot(hist, color='red')
    plt.plot(cdfNorm, color='blue')
    plt.xlim([0,256])

def process_adaptive_threshold(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    images = sorted([f for f in os.listdir(input_folder) if f.lower().endswith('.png')])

    results = []
    for fname in images:
        in_path = os.path.join(input_folder, fname)
        morphOpenImg = cv.imread(in_path, cv.IMREAD_GRAYSCALE)
        if morphOpenImg is None:
            continue

        # Adaptive Threshold
        adaptThreshImg = cv.adaptiveThreshold(
            morphOpenImg, 255,
            cv.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv.THRESH_BINARY,
            11, 2
        )

        # Save
        out_name = fname
        out_path = os.path.join(output_folder, out_name)
        cv.imwrite(out_path, adaptThreshImg)

        results.append((out_name, adaptThreshImg))

if __name__ == '__main__':
    input_folder = os.path.join(segmentation_folder, 'Morphological_Opening_Day-5')
    output_folder = os.path.join(segmentation_folder, 'Adaptive_Threshold_Day-5')
    process_adaptive_threshold(input_folder, output_folder)

**8. Complement Image**

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import os

def plotHist(image):
    hist = cv.calcHist([image],[0],None,[256],[0,256])
    cdf = hist.cumsum()
    cdfNorm = cdf * float(hist.max()) / cdf.max()
    plt.plot(hist, color='red')
    plt.plot(cdfNorm, color='blue')
    plt.xlim([0,256])

def process_complement(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    images = sorted([f for f in os.listdir(input_folder) if f.lower().endswith('.png')])

    for fname in images:
        in_path = os.path.join(input_folder, fname)
        img = cv.imread(in_path, cv.IMREAD_GRAYSCALE)
        if img is None:
            continue

        # Complement Image
        complementImg = cv.bitwise_not(img)

        # Save to output_folder
        out_name = fname
        out_path = os.path.join(output_folder, out_name)
        cv.imwrite(out_path, complementImg)

if __name__ == '__main__':
    input_dir = os.path.join(segmentation_folder, 'Adaptive_Threshold_Day-5')
    output_dir = os.path.join(segmentation_folder, 'Complement_Day-5')
    process_complement(input_dir, output_dir)

**9. Masking/Erosion**

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import os

def plotHist(image):
    hist = cv.calcHist([image],[0],None,[256],[0,256])
    cdf = hist.cumsum()
    cdfNorm = cdf * float(hist.max()) / cdf.max()
    plt.plot(hist, color='red')
    plt.plot(cdfNorm, color='blue')
    plt.xlim([0,256])

def remove_outer_outline(binary_img, erode_iters=5, kernel_size=7):
    contours, _ = cv.findContours(binary_img, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    if not contours:
        return binary_img

    largest = max(contours, key=cv.contourArea)

    # Buat mask area dalam kontur
    mask = np.zeros_like(binary_img)
    cv.drawContours(mask, [largest], -1, 255, thickness=cv.FILLED)

    # Erosi mask untuk menghapus tepi outline
    erode_kernel = cv.getStructuringElement(cv.MORPH_ELLIPSE, (kernel_size, kernel_size))
    inner_mask = cv.erode(mask, erode_kernel, iterations=erode_iters)

    # Terapkan mask ke hasil complement
    cleaned = cv.bitwise_and(binary_img, inner_mask)

    return cleaned

def process_masking_erosion(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    images = sorted([f for f in os.listdir(input_folder) if f.lower().endswith('.png')])

    for fname in images:
        in_path = os.path.join(input_folder, fname)
        img = cv.imread(in_path, cv.IMREAD_GRAYSCALE)
        if img is None:
            continue

        # Masking + Erosion
        contourRemoved = remove_outer_outline(img, erode_iters=5, kernel_size=7)

        # Simpan hasil ke output_folder
        out_name = fname
        out_path = os.path.join(output_folder, out_name)
        cv.imwrite(out_path, contourRemoved)

if __name__ == '__main__':
    input_folder = os.path.join(segmentation_folder, 'Complement_Day-5')
    output_folder = os.path.join(segmentation_folder, 'Masking_Erosion_Day-5')
    process_masking_erosion(input_folder, output_folder)

**10. Frangi Vesselness**

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import os

def plotHistCDF(image, subplot_index, title):
    hist = cv.calcHist([image],[0],None,[256],[0,256])
    cdf = hist.cumsum()
    cdfNorm = cdf * float(hist.max()) / cdf.max()
    plt.subplot(subplot_index[0], subplot_index[1], subplot_index[2])
    plt.plot(hist, color='red')
    plt.plot(cdfNorm, color='blue')
    plt.xlim([0,256])
    plt.xlabel(title)

def vesselEnhAll(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)

    images = sorted([f for f in os.listdir(input_folder) if f.lower().endswith('.png')])
    total_input = len(images)
    total_processed = 0
    results = []

    # Process all files
    for filename in images:
        imgPath = os.path.join(input_folder, filename)
        img = cv.imread(imgPath, cv.IMREAD_GRAYSCALE)
        if img is None:
            continue

        # Apply CLAHE
        claheObj = cv.createCLAHE(clipLimit=5, tileGridSize=(8,8))
        claheImg = claheObj.apply(img)

        # Frangi Vesselness
        claheNorm = claheImg / 255.0
        _, mask = cv.threshold(img, 30, 1, cv.THRESH_BINARY)
        frangiImg = frangi(claheNorm, sigmas=range(3,16,2))
        frangiMasked = frangiImg * mask
        frangiMasked8 = (frangiMasked*255).astype(np.uint8)

        # Save result
        cv.imwrite(os.path.join(output_folder, f"{filename}"), frangiMasked8)
        total_processed += 1
        results.append((filename, img.shape, frangiMasked8))

if __name__ == '__main__':
    input_folder = os.path.join(preprocessing_folder, 'Multiplication')
    output_folder = os.path.join(segmentation_folder, 'Frangi_Vesselness_Day-5')
    vesselEnhAll(input_folder, output_folder)

**11. Enhanced Vessel Visibility**

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import os

def plotHistCDF(image, subplot_index, title):
    hist = cv.calcHist([image],[0],None,[256],[0,256])
    cdf = hist.cumsum()
    cdfNorm = cdf * float(hist.max()) / cdf.max()
    plt.subplot(subplot_index[0], subplot_index[1], subplot_index[2])
    plt.plot(hist, color='red')
    plt.plot(cdfNorm, color='blue')
    plt.xlim([0,256])
    plt.xlabel(title)

def vesselEnhAll(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)

    images = sorted([f for f in os.listdir(input_folder) if f.lower().endswith('.png')])
    total_input = len(images)
    total_processed = 0
    results = []

    # Process all files
    for filename in images:
        imgPath = os.path.join(input_folder, filename)
        img = cv.imread(imgPath, cv.IMREAD_GRAYSCALE)
        if img is None:
            continue

        # Apply CLAHE
        claheObj = cv.createCLAHE(clipLimit=5, tileGridSize=(8,8))
        claheImg = claheObj.apply(img)

        # Frangi Vesselness
        claheNorm = claheImg / 255.0
        _, mask = cv.threshold(img, 30, 1, cv.THRESH_BINARY)
        frangiImg = frangi(claheNorm, sigmas=range(3,16,2))
        frangiMasked = frangiImg * mask
        frangiMasked8 = (frangiMasked*255).astype(np.uint8)

        # Enhanced Vessel Visibility
        claheEnh = cv.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
        vesselEnh = claheEnh.apply(frangiMasked8)

        # Save result
        cv.imwrite(os.path.join(output_folder, f"{filename}"), vesselEnh)
        total_processed += 1
        results.append((filename, img.shape, vesselEnh))

if __name__ == '__main__':
    input_folder = os.path.join(preprocessing_folder, 'Multiplication')
    output_folder = os.path.join(segmentation_folder, 'Frangi_Vessel_Visibility_Day-5')
    vesselEnhAll(input_folder, output_folder)